In [12]:
import os
from glob import glob
from IPython.display import display, HTML
import html
import pandas as pd
from math import ceil

# Memory Mapping Data
memory_mapping = [
    {"section": "TI.persistent", "start": 0x00007C94, "length": 0x0000C42, "details": "*"},
    {"section": "us.obj (.TI.persistent:gNVMemBlock)", "start": 0x00007C94, "length": 0x0000A50, "details": ""},
    {"section": "us.obj (.TI.persistent:ussAlgConfig)", "start": 0x00086E4, "length": 0x000008A, "details": ""},
    {"section": "processing.obj (.TI.persistent:gs_processing)", "start": 0x000876E, "length": 0x0000086, "details": ""},
    {"section": "USS_Lib_HAL.obj (.TI.persistent)", "start": 0x00087F4, "length": 0x0000038, "details": ""},
    {"section": "us.obj (.TI.persistent:gUssSWConfig)", "start": 0x000882C, "length": 0x0000022, "details": ""},
    {"section": "us.obj (.TI.persistent:ussMeasurementConfig)", "start": 0x000884E, "length": 0x0000022, "details": ""},
    {"section": "us.obj (.TI.persistent:ussCaptureConfig)", "start": 0x0008870, "length": 0x000001A, "details": ""},
    {"section": "us.obj (.TI.persistent:ussPulseConfig)", "start": 0x000888A, "length": 0x0000014, "details": ""},
    {"section": "us.obj (.TI.persistent:ussMeterConfig)", "start": 0x000889E, "length": 0x0000010, "details": ""},
    {"section": "us.obj (.TI.persistent:ussPLLConfig)", "start": 0x00088AE, "length": 0x000000C, "details": ""},
    {"section": "us.obj (.TI.persistent:ussSystemConfig)", "start": 0x00088BA, "length": 0x000000C, "details": ""},
    {"section": "systick.obj (.TI.persistent)", "start": 0x00088C6, "length": 0x00000008, "details": ""},
    {"section": "us.obj (.TI.persistent:singDualToneConfig)", "start": 0x00088CE, "length": 0x00000004, "details": ""},
    {"section": "us.obj (.TI.persistent:ussInterruptConfig)", "start": 0x00088D2, "length": 0x00000002, "details": ""},
    {"section": "us.obj (.TI.persistent:ussTriggerConfig)", "start": 0x00088D4, "length": 0x00000002, "details": ""},
]

# Function to determine which section a byte address belongs to
def get_section(address):
    for section in memory_mapping:
        if section["start"] <= address < section["start"] + section["length"]:
            return section["section"]
    return "Unknown Section"

# Function to read and parse a .txt file into a bytearray
def parse_txt_to_bytes(file_path, max_bytes):
    byte_array = bytearray()
    with open(file_path, 'r') as file:
        for line in file:
            # Remove any leading/trailing whitespace and split by spaces
            parts = line.strip().split()
            # Filter out any non-hex parts (e.g., addresses like @4000)
            hex_bytes = [part for part in parts if all(c in '0123456789ABCDEFabcdef' for c in part)]
            # Convert hex strings to bytes and append to byte_array
            for hb in hex_bytes:
                if len(byte_array) >= max_bytes:
                    break
                try:
                    byte = int(hb, 16)
                    byte_array.append(byte)
                except ValueError:
                    # Skip invalid hex representations
                    continue
            if len(byte_array) >= max_bytes:
                break
    return byte_array

# Function to generate a visually appealing HTML table with differences highlighted in red
def generate_pretty_html(reference_bytes, compare_bytes, max_bytes, compare_filename):
    bytes_per_line = 16
    rows = []
    first_diff_address = None
    total_diffs = 0
    sections_with_diffs = {}
    differences_details = {}

    for i in range(0, max_bytes, bytes_per_line):
        address = f'0x{i:04X}'
        ref_line = reference_bytes[i:i+bytes_per_line]
        cmp_line = compare_bytes[i:i+bytes_per_line]

        ref_hex = ' '.join(f'{b:02X}' for b in ref_line)
        cmp_hex_parts = []
        for j in range(len(cmp_line)):
            current_address = i + j
            section = get_section(current_address)
            if i + j >= len(reference_bytes):
                # Extra bytes in comparison
                cmp_hex_parts.append(f'<span style="color: red;">{cmp_line[j]:02X}</span>')
                if first_diff_address is None:
                    first_diff_address = current_address
                total_diffs += 1
                sections_with_diffs[section] = sections_with_diffs.get(section, 0) + 1
                differences_details.setdefault(section, []).append({
                    "address": f'0x{current_address:04X}',
                    "reference": "N/A",
                    "comparison": f'{cmp_line[j]:02X}'
                })
            elif cmp_line[j] != reference_bytes[i + j]:
                cmp_hex_parts.append(f'<span style="color: red;">{cmp_line[j]:02X}</span>')
                if first_diff_address is None:
                    first_diff_address = current_address
                total_diffs += 1
                sections_with_diffs[section] = sections_with_diffs.get(section, 0) + 1
                differences_details.setdefault(section, []).append({
                    "address": f'0x{current_address:04X}',
                    "reference": f'{reference_bytes[i + j]:02X}',
                    "comparison": f'{cmp_line[j]:02X}'
                })
            else:
                cmp_hex_parts.append(f'{cmp_line[j]:02X}')
        cmp_hex = ' '.join(cmp_hex_parts)

        rows.append({'Address': address, 'Reference': ref_hex, 'Comparison': cmp_hex})

    # Create a DataFrame for better styling with Pandas
    df = pd.DataFrame(rows)

    # Define a function to apply styles using Styler.map
    styled_df = df.style.map(lambda val: 'background-color: #ffe6e6;' if '<span style="color: red;">' in val else '', subset=['Comparison']).set_properties(**{
        'font-family': 'Courier New',
        'font-size': '12px'
    }).set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#f2f2f2'), ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '5px')]}
    ])

    # Convert styled DataFrame to HTML
    html_table = styled_df.to_html(escape=False)  # Use escape=False to allow HTML in the cells

    # Prepare the first difference address in hexadecimal
    if first_diff_address is not None:
        first_diff_hex = f'0x{first_diff_address:04X}'
    else:
        first_diff_hex = 'No differences found within the compared range.'

    # Generate a summary table of sections with differences
    if sections_with_diffs:
        sections_summary_rows = []
        for section, count in sorted(sections_with_diffs.items(), key=lambda x: x[0]):
            sections_summary_rows.append({'Section': section, 'Differences': count})
        sections_summary_df = pd.DataFrame(sections_summary_rows)
        sections_summary_styled = sections_summary_df.style.set_properties(**{
            'font-family': 'Arial',
            'font-size': '12px'
        }).set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#d9edf7'), ('text-align', 'center')]},
            {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '5px')]}
        ])
        sections_summary_html = sections_summary_styled.to_html(index=False)
    else:
        sections_summary_html = "<p>No sections with differences.</p>"

    # Generate detailed differences per section (collapsible sections)
    if differences_details:
        details_html = ""
        for section, diffs in differences_details.items():
            # Limit the number of detailed differences displayed per section to prevent HTML overload
            max_display = 50  # Adjust as needed
            display_diffs = diffs[:max_display]
            remaining = len(diffs) - max_display
            diffs_table = pd.DataFrame(display_diffs)
            diffs_styled = diffs_table.style.set_properties(**{
                'font-family': 'Courier New',
                'font-size': '12px'
            }).set_table_styles([
                {'selector': 'th', 'props': [('background-color', '#fcf8e3'), ('text-align', 'center')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '5px')]}
            ]).hide(axis='index')  # Updated method

            diffs_html = diffs_styled.to_html(index=False)
            if remaining > 0:
                diffs_html += f"<p>And {remaining} more differences in this section...</p>"
            # Create a collapsible section
            details_html += f"""
            <details>
                <summary><strong>{html.escape(section)}</strong> - {len(diffs)} differences</summary>
                {diffs_html}
            </details>
            <br/>
            """
    else:
        details_html = "<p>No detailed differences to display.</p>"

    # Combine the table with a header indicating the first difference and total differences
    full_html = f"""
    <div style="border: 1px solid #ccc; border-radius: 5px; padding: 10px; margin: 5px; width: 100%; box-sizing: border-box;">
        <h4 style="margin-top: 0;">Comparison: {html.escape(compare_filename)} vs Reference</h4>
        <p><strong>First difference at:</strong> {first_diff_hex}</p>
        <p><strong>Total differences:</strong> {total_diffs}</p>
        <h5>Sections with Differences:</h5>
        {sections_summary_html}
        <h5>Detailed Differences per Section:</h5>
        {details_html}
        <h5>Hexadecimal Comparison:</h5>
        <div style="max-height: 400px; overflow: auto;">
            {html_table}
        </div>
    </div>
    """
    return full_html

# Main comparison function
def compare_txt_files_enhanced(folder_path='.', max_bytes=0x4A38, max_comparisons=12):
    # Get list of .txt files in the specified folder (no subfolders)
    txt_files = [f for f in glob(os.path.join(folder_path, '*.txt')) if os.path.isfile(f)]
    
    if not txt_files:
        print("No .txt files found in the specified directory.")
        return
    
    # Sort files alphanumerically
    txt_files_sorted = sorted(txt_files)
    
    # Use the first file as reference
    reference_file = txt_files_sorted[0]
    print(f"Reference File: {os.path.basename(reference_file)}")
    
    reference_bytes = parse_txt_to_bytes(reference_file, max_bytes)
    
    # Check if reference_bytes has enough data
    if len(reference_bytes) < max_bytes:
        print(f"Warning: Reference file has only {len(reference_bytes)} bytes, expected {max_bytes}.")
    
    # Limit the number of comparison files to max_comparisons
    comparison_files = txt_files_sorted[1:max_comparisons+1]
    if len(txt_files_sorted) -1 > max_comparisons:
        print(f"Note: Only the first {max_comparisons} comparison files will be processed.")
    
    # Initialize HTML content with a grid layout
    num_columns = 3  # Number of columns in the grid
    grid_template_columns = '30% 30% 30%'  # Adjust widths as needed
    
    all_comparisons_html = f"""
    <h2>Comparison Report</h2>
    <p><strong>Reference File:</strong> {html.escape(os.path.basename(reference_file))}</p>
    <div style="display: grid; grid-template-columns: {grid_template_columns}; grid-gap: 10px;">
    """

    # Iterate over comparison files and generate HTML for each
    for compare_file in comparison_files:
        print(f"Comparing {os.path.basename(compare_file)} to reference...")
        compare_bytes = parse_txt_to_bytes(compare_file, max_bytes)
        
        # Check if compare_bytes has enough data
        if len(compare_bytes) < max_bytes:
            print(f"Warning: {os.path.basename(compare_file)} has only {len(compare_bytes)} bytes, expected {max_bytes}.")
        
        # Generate HTML comparison
        diff_html = generate_pretty_html(reference_bytes, compare_bytes, max_bytes, os.path.basename(compare_file))
        
        # Append to the grid
        all_comparisons_html += diff_html
    
    all_comparisons_html += "</div>"  # Close the grid div
    
    # Display the combined HTML
    display(HTML(all_comparisons_html))

# Usage
# You can specify the folder_path if the .txt files are not in the current directory
# Set max_comparisons=12 to handle up to 12 comparison files
compare_txt_files_enhanced(folder_path='.', max_bytes=0x4500, max_comparisons=12)  # Replace '.' with your target directory if needed


Reference File: fcx_memory_saving.txt
Comparing fcx_msp_20241031_0948.txt to reference...
Comparing fcx_msp_secdep.txt to reference...


,Address,Reference,Comparison
0,0x0000,6A 14 B1 00 28 00 8C 00 98 90 B0 13 64 7C 71 0C,6A 14 B1 00 28 00 8C 00 98 90 B0 13 64 7C 71 0C
1,0x0010,20 00 8C 00 98 90 B0 13 34 7C 71 0C 24 00 0F 43,20 00 8C 00 98 90 B0 13 34 7C 71 0C 24 00 0F 43
2,0x0020,81 4F 04 00 81 4F 06 00 81 4F 08 00 81 4F 0A 00,81 4F 04 00 81 4F 06 00 81 4F 08 00 81 4F 0A 00
3,0x0030,81 4F 0C 00 81 4F 0E 00 B1 40 C6 07 02 00 03 43,81 4F 0C 00 81 4F 0E 00 B1 40 C6 07 02 00 03 43
4,0x0040,2F 00 A0 90 3F 0F 08 00 BF 40 80 84 06 00 BF 40,2F 00 A0 90 3F 0F 08 00 BF 40 80 84 06 00 BF 40
5,0x0050,1E 00 08 00 8C 00 98 90 B0 13 C0 4C 8C 00 98 90,1E 00 08 00 8C 00 98 90 B0 13 C0 4C 8C 00 98 90
6,0x0060,B0 13 DC 4D 2F 00 A8 90 71 0F 1C 00 04 43 05 43,B0 13 DC 4D 2F 00 A8 90 71 0F 1C 00 04 43 05 43
7,0x0070,06 43 07 43 1F 4F 02 00 0F 93 20 24 3E 01 20 00,06 43 07 43 1F 4F 02 00 0F 93 20 24 3E 01 20 00
8,0x0080,71 0E 18 00 81 4F 00 00 3F 01 18 00 3C 4F 71 0F,71 0E 18 00 81 4F 00 00 3F 01 18 00 3C 4F 71 0F
9,0x0090,18 00 1C 93 10 38 B1 13 22 1C C8 0C C9 0D CA 0E,18 00 1C 93 10 38 B1 13 22 1C C8 0C C9 0D CA 0E


In [1]:
import re
import pandas as pd

# Sample data snippet (you should replace this with your full data)
data = """
.text      0    00004000    00003c94     us.obj (.text:us_trimFrequencySwap)
.data      0    00007c94    00000838     RUN ADDR = 00002c00
.bss       0    0000217a    0000035e     screenmng.obj (.bss:gs_screenManager)
"""

# Regular expression to match each line
pattern = re.compile(r'(\.\w+)\s+\*\s+0\s+([0-9a-fA-F]+)\s+([0-9a-fA-F]+)\s+(.+)')

# Lists to store parsed data
sections = []
origins = []
lengths = []
inputs = []

# Split data into lines and parse
for line in data.strip().split('\n'):
    match = pattern.match(line)
    if match:
        section, page, origin, length_input = match.groups()
        # Further split length and input sections
        length_match = re.match(r'([0-9a-fA-F]+)\s+(.+)', length_input.strip())
        if length_match:
            length, input_section = length_match.groups()
            sections.append(section)
            origins.append(int(origin, 16))
            lengths.append(int(length, 16))
            inputs.append(input_section)
    else:
        # Handle lines without '*' (if any)
        pass

# Create DataFrame
df = pd.DataFrame({
    'Section': sections,
    'Origin': origins,
    'Length': lengths,
    'Input Section': inputs
})

# Sort by Origin
df = df.sort_values('Origin').reset_index(drop=True)

# Calculate End Address
df['End'] = df['Origin'] + df['Length']

# Display the DataFrame
print(df)


Empty DataFrame
Columns: [Section, Origin, Length, Input Section, End]
Index: []
